# 🤖 Módulo 3 · Cuaderno 1 — El equipo de Nexo
### Orquestación multiagente, puntos de control y colaboración humano-IA

**AVE Ingeniería de Prompting y Agentes Autónomos · Universidad Ean**

Nexo está saturado: atiende precios, recogidas, pagos y quejas a la vez, y sus instrucciones ya son demasiado largas. Sam se pregunta si Nexo necesita **un equipo de agentes** o si le basta con ordenar su trabajo. Y Lucía, la analista de pagos, necesita estar segura de que ningún pago grande sale sin su aprobación.

**Al terminar podrás:**
1. Decidir cuándo vale la pena usar varios agentes.
2. Programar una clase `Agente` reutilizable.
3. Implementar tres patrones de coordinación: **enrutador**, **secuencial (redactor → revisor)** y **orquestador-trabajadores**.
4. Insertar un **punto de control humano** (*human-in-the-loop*) y un **protocolo de escalamiento**.

> Relación con el AVE: Módulo 3, Págs. 01 a 05.

## 0. Antes de empezar: cómo usar este cuaderno

Un **cuaderno de Colab** mezcla texto (como este) y **celdas de código** que puedes ejecutar.

- Para ejecutar una celda, haz clic en ella y presiona **Shift + Enter** (o el botón ▶ de la izquierda).
- Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla, vuelve a ejecutar desde la sección de configuración.
- Lo que escribe el código aparece debajo de la celda: eso es la **salida**.
- Si cierras la pestaña o pasa mucho tiempo, Colab reinicia la sesión y hay que ejecutar todo otra vez (*Entorno de ejecución → Ejecutar todas*).

### Tu clave de Gemini (gratis)

Nexo usa los modelos **Gemini** de Google a través de su API. Necesitas una clave personal:

1. Entra a <https://aistudio.google.com/apikey> con tu cuenta de Google y haz clic en **Create API key**.
2. Copia la clave.
3. En Colab, abre el panel izquierdo y haz clic en el ícono de la **llave 🔑 (Secretos)**.
4. Agrega un secreto con nombre `GOOGLE_API_KEY`, pega la clave en *Valor* y activa **Acceso al cuaderno**.

> ⚠️ **Nunca escribas tu clave dentro del código** ni la compartas: si entregas o compartes el cuaderno, cualquiera podría usarla. Los *Secretos* de Colab la guardan aparte.

La capa gratuita de Gemini tiene un límite de solicitudes por minuto. Si ves un error **429**, espera un minuto: el código de este cuaderno ya reintenta solo.

## 1. Configuración

In [ ]:
# ── Instalación de librerías ─────────────────────────────────────────────
# El signo "!" ejecuta un comando de la terminal dentro de Colab.
# google-genai : la librería oficial de Google para usar los modelos Gemini.
# pydantic     : sirve para definir "moldes" de datos y pedir respuestas con estructura fija.
# -q (quiet)   : instala sin llenar la pantalla de mensajes.
!pip install -q -U google-genai pydantic

In [ ]:
# ── Configuración: conectar con Gemini ───────────────────────────────────
import time                      # para medir tiempos y esperar entre reintentos
import json                      # para leer y escribir datos en formato JSON
from google import genai         # cliente para hablar con los modelos Gemini
from google.genai import types   # "tipos": objetos de configuración (temperatura, herramientas, etc.)


def obtener_clave():
    """Lee la clave desde los Secretos de Colab; si no existe, la pide por teclado."""
    try:
        from google.colab import userdata
        clave = userdata.get("GOOGLE_API_KEY")
        if clave:
            return clave
    except Exception:
        pass  # no estamos en Colab o el secreto no tiene acceso al cuaderno
    from getpass import getpass  # getpass oculta lo que escribes
    return getpass("Pega tu clave de Gemini y presiona Enter: ")


# El "cliente" es el objeto que envía nuestras preguntas al modelo y trae las respuestas.
client = genai.Client(api_key=obtener_clave())

# Los nombres de los modelos cambian con el tiempo. Probamos una lista en orden de preferencia
# y nos quedamos con el primero que esté disponible para tu clave.
MODELOS_PREFERIDOS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash"]


def elegir_modelo(preferidos=MODELOS_PREFERIDOS):
    disponibles = []
    for m in client.models.list():
        acciones = m.supported_actions or []
        if "generateContent" in acciones:
            disponibles.append(m.name.replace("models/", ""))
    for nombre in preferidos:
        if nombre in disponibles:
            return nombre
    # Si ninguno de la lista existe, usamos cualquier modelo "flash" disponible.
    flash = [n for n in disponibles if "flash" in n and "image" not in n and "tts" not in n]
    return flash[0] if flash else disponibles[0]


MODELO = elegir_modelo()
print("✅ Conectado. Modelo que usará Nexo:", MODELO)


def llamar_con_reintentos(funcion, *args, intentos=5, **kwargs):
    """Ejecuta una llamada a la API y, si el servicio está saturado (error 429 o 503),
    espera y vuelve a intentar. Así el cuaderno no se detiene por límites de la capa gratuita."""
    espera = 10
    for intento in range(1, intentos + 1):
        try:
            return funcion(*args, **kwargs)
        except Exception as error:
            mensaje = str(error)
            if any(codigo in mensaje for codigo in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and intento < intentos:
                print(f"⏳ Servicio ocupado (intento {intento}). Espero {espera} s y reintento...")
                time.sleep(espera)
                espera *= 2  # cada vez esperamos el doble
            else:
                raise


def preguntar(prompt, sistema=None, temperatura=None, modelo=None, **otros):
    """Envía un texto al modelo y devuelve la respuesta como texto.
    - prompt:      lo que le pedimos al modelo.
    - sistema:     instrucciones de sistema (quién es y cómo se comporta Nexo).
    - temperatura: 0 = respuestas más estables; 1 o más = respuestas más variadas.
    - otros:       cualquier otro ajuste de GenerateContentConfig."""
    config = types.GenerateContentConfig(system_instruction=sistema, temperature=temperatura, **otros)
    respuesta = llamar_con_reintentos(
        client.models.generate_content, model=modelo or MODELO, contents=prompt, config=config
    )
    return respuesta.text


print(preguntar("Saluda en una sola frase, en español."))

In [ ]:
# ── Datos (ficticios) de la Cooperativa Raíces ───────────────────────────
# Son la "fuente de verdad" de la cooperativa. Nexo no debe inventar nada que no esté aquí.
COOPERATIVA = {
    "nombre": "Cooperativa Raíces",
    "producto": "café pergamino seco",
    "precio_referencia_carga": 2_150_000,   # pesos colombianos por carga de 125 kg (se actualiza cada lunes)
    "fecha_precio": "lunes de esta semana",
    "horario_acopio": "lunes a sábado, de 6:00 a. m. a 2:00 p. m.",
}

REGLAMENTO = [
    "Art. 1. Precio de referencia: la cooperativa publica cada lunes el precio de referencia por carga de 125 kg de café pergamino seco. Nexo solo informa el precio vigente publicado.",
    "Art. 2. Calidad: el café se recibe con humedad entre 10 % y 12 %. Si la humedad supera el 12 %, se descuenta el 1 % del valor por cada punto adicional.",
    "Art. 3. Horario del centro de acopio: lunes a sábado de 6:00 a. m. a 2:00 p. m. Los domingos y festivos no hay recepción.",
    "Art. 4. Recogida en camión: solo para entregas de 10 cargas o más, en veredas con vía carreteable, y se solicita con 48 horas de anticipación.",
    "Art. 5. Pagos: los pagos se hacen por transferencia a la cuenta registrada del asociado dentro de los 3 días hábiles siguientes a la entrega.",
    "Art. 6. Aprobación de pagos: todo pago igual o superior a 5.000.000 de pesos requiere la aprobación de la analista de pagos.",
    "Art. 7. Anticipos de cosecha: un asociado puede pedir un anticipo de hasta el 30 % del valor entregado el año anterior.",
    "Art. 8. Datos personales: Nexo no comparte datos de un asociado con otra persona y nunca pide contraseñas ni claves bancarias.",
    "Art. 9. Quejas: toda queja se registra con un número de caso y la responde una persona del equipo en máximo 5 días hábiles.",
    "Art. 10. Asamblea: la asamblea general de asociados se realiza el último sábado de marzo.",
]

ASOCIADOS = {
    "A-101": {"nombre": "Aurelio Gómez", "vereda": "La Esperanza", "via_carreteable": True,  "cargas_anio_anterior": 48},
    "A-102": {"nombre": "Rosa Pineda",   "vereda": "El Diviso",    "via_carreteable": False, "cargas_anio_anterior": 22},
    "A-103": {"nombre": "Julián Ortiz",  "vereda": "Alto Bonito",  "via_carreteable": True,  "cargas_anio_anterior": 9},
}

print("Precio de referencia:", f"${COOPERATIVA['precio_referencia_carga']:,}".replace(",", "."), "por carga")
print("Artículos del reglamento:", len(REGLAMENTO), "| Asociados registrados:", len(ASOCIADOS))

### Modo interactivo
Cuando un agente necesita la aprobación de una persona, el cuaderno te la pedirá con un cuadro de texto (`input`). Si vas a ejecutar todo de corrido (por ejemplo, para guardar el PDF), pon `MODO_INTERACTIVO = False` y la aprobación se **simula** con una regla escrita.

In [ ]:
MODO_INTERACTIVO = False   # cambia a True para aprobar o rechazar tú mismo

## 2. ¿Uno o varios agentes?

Más agentes **no** siempre es mejor: cada uno agrega llamadas, costo, tiempo y puntos de falla. Usa varios agentes cuando:

- las tareas requieren **instrucciones o herramientas muy distintas**;
- quieres que un agente **revise** el trabajo de otro;
- las tareas se pueden hacer **en paralelo**.

Si un solo agente con buenas herramientas resuelve el problema, quédate con uno.

## 3. Una clase `Agente` reutilizable

Una **clase** es un molde para crear objetos. Cada agente tendrá su nombre, su rol (system prompt) y sus herramientas.

In [ ]:
class Agente:
    """Un agente = un modelo + un rol + (opcionalmente) herramientas."""

    def __init__(self, nombre, rol, herramientas=None, temperatura=0.2):
        self.nombre = nombre
        self.config = types.GenerateContentConfig(
            system_instruction=rol, tools=herramientas or None, temperature=temperatura)

    def responder(self, mensaje):
        r = llamar_con_reintentos(client.models.generate_content, model=MODELO,
                                  contents=mensaje, config=self.config)
        return r.text.strip()

    def __repr__(self):
        return f"<Agente {self.nombre}>"

In [ ]:
# Herramientas (las mismas ideas del Módulo 2, resumidas)
def consultar_precio_referencia() -> dict:
    """Devuelve el precio de referencia vigente en pesos por carga de 125 kg."""
    return {"precio_por_carga": COOPERATIVA["precio_referencia_carga"]}

def verificar_recogida(codigo: str, cargas: int) -> dict:
    """Verifica si aplica la recogida en camión (Art. 4: 10 cargas o más y vía carreteable)."""
    a = ASOCIADOS.get(codigo.upper())
    if not a:
        return {"error": "Asociado no encontrado"}
    return {"aplica": cargas >= 10 and a["via_carreteable"], "vereda": a["vereda"],
            "via_carreteable": a["via_carreteable"], "cargas": cargas}

def calcular_liquidacion(cargas: int, humedad: float) -> dict:
    """Calcula el pago de una entrega con descuento del 1 % por punto de humedad sobre 12 % (Art. 2)."""
    bruto = cargas * COOPERATIVA["precio_referencia_carga"]
    neto = round(bruto * (1 - max(0.0, humedad - 12) / 100))
    return {"bruto": bruto, "neto": neto, "requiere_aprobacion": neto >= 5_000_000}

REGLAS = "Reglamento:\n" + "\n".join(REGLAMENTO)

agente_precios = Agente("Precios", "Eres el especialista en precios de la Cooperativa Raíces. Usa tu herramienta; nunca inventes cifras.\n" + REGLAS,
                        [consultar_precio_referencia])
agente_logistica = Agente("Logística", "Eres el especialista en recogidas en camión. Verifica siempre con tu herramienta antes de responder.\n" + REGLAS,
                          [verificar_recogida])
agente_pagos = Agente("Pagos", "Eres el especialista en liquidaciones y pagos. Calcula siempre con tu herramienta.\n" + REGLAS,
                      [calcular_liquidacion])
agente_general = Agente("General", "Eres Nexo, asistente general de la Cooperativa Raíces. Responde con el reglamento; si no sabes, escala a una persona.\n" + REGLAS)

print(agente_logistica.responder("Soy A-103, tengo 11 cargas. ¿Me recogen en camión?"))

## 4. Patrón 1: enrutador (*router*)

Un agente **enrutador** lee el mensaje y decide **a qué especialista** enviarlo. Usamos una salida estructurada para que la decisión sea siempre una de las opciones válidas.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Ruta(BaseModel):
    especialista: Literal["Precios", "Logística", "Pagos", "General"]
    motivo: str = Field(description="Por qué se eligió este especialista, en una frase")

EQUIPO = {a.nombre: a for a in [agente_precios, agente_logistica, agente_pagos, agente_general]}

def enrutar(mensaje):
    config = types.GenerateContentConfig(
        system_instruction="Eres el enrutador de Nexo. Elige el especialista adecuado para el mensaje.",
        response_mime_type="application/json", response_schema=Ruta, temperature=0)
    r = llamar_con_reintentos(client.models.generate_content, model=MODELO, contents=mensaje, config=config)
    return r.parsed or Ruta.model_validate_json(r.text)

for m in ["¿A cómo está la carga esta semana?",
          "Soy A-102, tengo 25 cargas, ¿me mandan el camión?",
          "¿Cuándo es la asamblea?"]:
    ruta = enrutar(m)
    print(f"👤 {m}\n🧭 → {ruta.especialista} ({ruta.motivo})\n🤖 {EQUIPO[ruta.especialista].responder(m)}\n")

## 5. Patrón 2: secuencial (redactor → revisor)

Un agente **redacta** y otro **revisa** contra el reglamento antes de enviar la respuesta. Es útil cuando un error es costoso. El revisor no ve las instrucciones del redactor: lo evalúa con "ojos frescos".

In [ ]:
class Revision(BaseModel):
    cumple: bool
    problemas: list[str]
    respuesta_corregida: str

revisor_config = types.GenerateContentConfig(
    system_instruction="Eres el revisor de calidad de la Cooperativa Raíces. Verifica que la respuesta cumpla el reglamento y no prometa nada que no esté permitido.\n" + REGLAS,
    response_mime_type="application/json", response_schema=Revision, temperature=0)

redactor_apresurado = Agente("Redactor", "Eres un asistente muy amable que siempre intenta complacer al asociado.", temperatura=0.9)

mensaje = "Soy Rosa de El Diviso, tengo 25 cargas. ¿Me mandan el camión el sábado?"
borrador = redactor_apresurado.responder(mensaje)
r = llamar_con_reintentos(client.models.generate_content, model=MODELO, config=revisor_config,
                          contents=f"Mensaje del asociado: {mensaje}\nRespuesta propuesta: {borrador}")
revision = r.parsed or Revision.model_validate_json(r.text)
print("✍️ Borrador:", borrador)
print("\n🔍 ¿Cumple?:", revision.cumple, "| Problemas:", revision.problemas)
print("\n✅ Respuesta final:", revision.respuesta_corregida if not revision.cumple else borrador)

## 6. Puntos de intervención: aprobación humana y escalamiento

No todas las decisiones deben tomarlas los agentes. Definimos **niveles de autonomía** (Parasuraman et al., 2000):

| Nivel | Quién decide | Ejemplo en Nexo |
|---|---|---|
| 1 · Autónomo | El agente actúa solo | Informar el precio o el horario |
| 2 · Con aprobación | El agente propone y una persona aprueba | Pagos ≥ 5 millones (Art. 6) → Lucía |
| 3 · Escalado | Una persona decide y responde | Quejas (Art. 9), casos no previstos |

In [ ]:
def pedir_aprobacion(propuesta, aprobador="Lucía (analista de pagos)"):
    """Punto de control humano. En modo interactivo pregunta a una persona; si no, aplica una regla simulada."""
    print(f"\n⏸️ PUNTO DE CONTROL → {aprobador}\n   Propuesta: {propuesta}")
    if MODO_INTERACTIVO:
        decision = input("   ¿Aprueba? (s/n): ").strip().lower() == "s"
    else:
        decision = propuesta.get("neto", 0) <= 20_000_000   # regla simulada: aprueba hasta 20 millones
        print("   (simulado)", "Aprobado" if decision else "Rechazado")
    return decision


def decidir_nivel(categoria, liquidacion=None):
    if categoria == "queja":
        return 3
    if liquidacion and liquidacion["requiere_aprobacion"]:
        return 2
    return 1


def procesar_pago(codigo, cargas, humedad):
    liq = calcular_liquidacion(cargas, humedad)
    nivel = decidir_nivel("pago", liq)
    if nivel == 1:
        return f"Nivel 1 · Pago de ${liq['neto']:,} programado automáticamente para {codigo}.".replace(",", ".")
    aprobado = pedir_aprobacion({"asociado": codigo, **liq})
    if aprobado:
        return f"Nivel 2 · Pago de ${liq['neto']:,} aprobado por Lucía y programado.".replace(",", ".")
    return "Nivel 2 · Pago rechazado: se escala a revisión manual y se informa al asociado."

print(procesar_pago("A-103", 2, 12))     # 4,3 millones → nivel 1 (automático)
print(procesar_pago("A-102", 3, 12))     # 6,45 millones → nivel 2 (en modo simulado se aprueba)
print(procesar_pago("A-101", 14, 13.5))  # 29,6 millones → nivel 2 (en modo simulado se rechaza)

## 7. Patrón 3: orquestador-trabajadores

El **orquestador** recibe un mensaje complejo, lo divide en subtareas, las reparte entre los especialistas y une las respuestas. Aquí combinamos todo: enrutamiento, especialistas y escalamiento.

In [ ]:
class Subtarea(BaseModel):
    especialista: Literal["Precios", "Logística", "Pagos", "General"]
    pregunta: str

class Descomposicion(BaseModel):
    es_queja: bool = Field(description="True si el mensaje expresa una queja o molestia")
    subtareas: list[Subtarea]

def orquestador(mensaje):
    config = types.GenerateContentConfig(
        system_instruction="Eres el orquestador de Nexo. Divide el mensaje en subtareas para los especialistas Precios, Logística, Pagos o General.",
        response_mime_type="application/json", response_schema=Descomposicion, temperature=0)
    r = llamar_con_reintentos(client.models.generate_content, model=MODELO, contents=mensaje, config=config)
    d = r.parsed or Descomposicion.model_validate_json(r.text)

    if d.es_queja:   # Nivel 3: una persona atiende las quejas (Art. 9)
        return "🙋 Nivel 3 · Su caso fue registrado y una persona del equipo le responderá en máximo 5 días hábiles."

    partes = []
    for s in d.subtareas:
        print(f"   🧩 {s.especialista}: {s.pregunta}")
        partes.append(EQUIPO[s.especialista].responder(s.pregunta))
    union = preguntar("Une estas respuestas en un solo mensaje claro de máximo 5 frases para el asociado:\n" + "\n".join(partes),
                      sistema="Eres Nexo.", temperatura=0.2)
    return union

print(orquestador("Soy A-101. ¿A cómo está la carga y me pueden recoger 15 cargas el jueves?"))
print()
print(orquestador("Estoy furioso, el camión no llegó y perdí la cosecha"))

## 8. Reto para tu proyecto 🎯

1. Decide y **justifica** si tu agente trabaja solo o en equipo. Si es un equipo, ¿qué patrón usa?
2. Dibuja el **diagrama de roles** (agentes, herramientas y personas).
3. Define la **matriz humano-agente**: qué decide el agente, qué aprueba una persona y qué se escala.
4. Implementa al menos **un punto de control humano** y pruébalo con un caso que deba detenerse. ¡Esa es la escena clave de la demostración en tu video!

**Lecturas de apoyo:** Dorri et al. (2018); Liang et al. (2024); Qian et al. (2024); Parasuraman et al. (2000); Mosqueira-Rey et al. (2023); Bansal et al. (2021).

## 📄 ¿Cómo guardar este cuaderno en PDF con todas las salidas?

1. *Entorno de ejecución → Reiniciar y ejecutar todas* (o *Ejecutar todas*). Espera a que todas las celdas terminen.
2. Revisa que cada celda muestre su salida y que no haya errores en rojo.
3. *Archivo → Imprimir* y, en *Destino*, elige **Guardar como PDF**.
4. Para el `.ipynb`: *Archivo → Descargar → Descargar .ipynb*.